# 🚀 Production Docker

**Deploy containers safely**

## 📋 Overview

**What you'll learn:**
- Security hardening
- Health checks
- Logging
- Secrets management

**Time estimate:** ⏱️ 50 minutes

## 🔒 Security Hardening

```dockerfile
# Production Dockerfile with security
FROM python:3.11-slim

# 1. Run as non-root user
RUN useradd -m -u 1000 appuser

# 2. Set secure permissions
WORKDIR /app
COPY --chown=appuser:appuser . .

# 3. Install only what's needed
RUN pip install --no-cache-dir -r requirements.txt

# 4. Remove package manager
RUN apt-get remove -y apt && \
    rm -rf /var/lib/apt/lists/*

# 5. Switch to non-root
USER appuser

# 6. Don't expose unnecessary ports
EXPOSE 8000

# 7. Health check
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

## 🔐 Secrets Management

### Bad:
```dockerfile
# ❌ Never do this!
ENV OPENAI_API_KEY=sk-abc123...
```

### Good:
```yaml
# docker-compose.yml
services:
  api:
    environment:
      - OPENAI_API_KEY  # Read from host env
    env_file:
      - .env  # Or from file (git-ignored)
    secrets:
      - openai_key  # Or use Docker secrets

secrets:
  openai_key:
    external: true
```

### Best (Production):
```bash
# Use secrets manager
docker run \
  -e OPENAI_API_KEY=$(aws secretsmanager get-secret-value --secret-id openai-key --query SecretString --output text) \
  llm-api
```

## 📊 Logging

```dockerfile
# Log to stdout/stderr (Docker best practice)
CMD ["uvicorn", "main:app", "--log-config", "logging.json"]
```

```python
# logging.json
{
  "version": 1,
  "handlers": {
    "console": {
      "class": "logging.StreamHandler",
      "stream": "ext://sys.stdout",
      "formatter": "json"
    }
  },
  "root": {
    "level": "INFO",
    "handlers": ["console"]
  }
}
```

### View logs:
```bash
# Real-time logs
docker logs -f container_id

# Last 100 lines
docker logs --tail 100 container_id

# With timestamps
docker logs -t container_id

# Log driver (send to external system)
docker run --log-driver=syslog llm-api
```

## 🏥 Health Checks

```dockerfile
# In Dockerfile
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1
```

```python
# In your FastAPI app
@app.get("/health")
async def health():
    # Check dependencies
    try:
        redis.ping()
        db.execute("SELECT 1")
        return {"status": "healthy"}
    except Exception as e:
        raise HTTPException(503, "Service unhealthy")
```

**Docker will:**
- Mark unhealthy containers
- Can auto-restart
- Load balancers skip unhealthy

## ✅ Summary

**Production checklist:**

- ✅ Run as non-root user
- ✅ Use secrets manager
- ✅ Health checks enabled
- ✅ Logs to stdout
- ✅ Resource limits set
- ✅ No sensitive data in image
- ✅ Security scanning enabled
- ✅ Automated backups

**Security scanning:**
```bash
# Scan for vulnerabilities
docker scan llm-api:latest

# Or use Trivy
trivy image llm-api:latest
```

### Congratulations! 🎉

You've completed the **Docker** module!

### Next: `14_model_serving/01_serving_basics.ipynb`